In [2]:
# operator overloading 예제

class Number:
    def __init__(self, value):
        self.value = value

    # add 메서드 역할 (__add__ 오버로딩)
    def __add__(self, other):
        return Number(self.value + other.value)

    def __str__(self):
        return str(self.value)

# 객체 생성
a = Number(10)
b = Number(20)

# 더하기(+) 연산자 적용
c = a + b

print("a = ", a)
print("b = ", b)
print("a + b = ", c)

a =  10
b =  20
a + b =  30


In [3]:
# a+b 를 실행하는 경우, 아래와 같은 코드가 호출된다.

c = a.__add__(b)
print(c)

30


In [5]:
# 2차원 좌표 객체를 더하는 경우를 고려하여 __add__ 메서드를 수정하면 

class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        return Point(self.x + other.x, self.y + other.y)

    def __str__(self):
        return f"({self.x}, {self.y})"

p1 = Point(1, 2)
p2 = Point(3, 4)

p3 = p1 + p2

print(p1)
print(p2)
print(p3)

(1, 2)
(3, 4)
(4, 6)


In [15]:
# 여러가지의 연산자를 추가해보자

import math

class Vector:
    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

    def __repr__(self):
        return f'Vector({self.x!r}, {self.y!r})'

    def __abs__(self):
        return math.hypot(self.x, self.y)  # 유클리드 노름(Euclidean norm) 계산

    def __bool__(self):
        return bool(abs(self))

    def __add__(self, other):
        x = self.x + other.x
        y = self.y + other.y
        return Vector(x, y)

    def __mul__(self, other):
        x = self.x * other
        y = self.y * other
        return Vector(x, y)

    def __mul__(self, scalar):
        return Vector(self.x * scalar, self.y * scalar)  
        # 두번째 __mul__()이 덮어쓰므로 처음 작성한 메서드가 동작하지 않는다. (260814)


In [16]:
v1 = Vector(1, 2)
v2 = Vector(3, 4)

v1 + v2

Vector(4, 6)

In [ ]:
v1 * v2    # 두번째 __mul__() 이 덮어쓰므로 처음에 정의된 벡터 곱 메서드가 동작하지 않음 (260814)

TypeError: unsupported operand type(s) for *: 'int' and 'Vector'

In [ ]:
v1 * 3   # 덮어쓴 메서드가 동작함

Vector(3, 6)

In [28]:
# @typing.overload()를 사용하는 방법 : 동일한 이름의 메서드를 타입에 따라 다르게 적용 가능 (260814)

'''
@typing.overload는 실제 런타임 동작을 바꾸는 기능이 아니라 타입 힌트(type hint)용이다.
따라서 여러 형태의  __mul()__() 시그니처를 정의해 IDE와 타입 검사기(mypy)가 이해하도록하고
실제 구현은 하나만 작성해야 한다.
'''

from __future__ import annotations 
# 파이썬이 모든 타입 힌트를 문자열처럼 취급한다. (사용하는 이유 설명)
# 클래스가 정의되는 시점에는 Vector 클래스가 아직 완전히 만들어지지 않았기 때문에 

from typing import overload

class Vector:
    def __init__(self, x: float, y: float):
        self.x = x
        self.y = y

    # 입력, 출력의 타입을 미리 정의함
    # Vector * Vector
    @overload
    def __mul__(self, other: Vector) -> Vector:   # 아직 Vector라는 형태가 완성되지 않은 문제가 있으므로
        ...

    # Vector * Scalar
    @overload
    def __mul__(self, other: int | float) -> Vector:
        ...

    # 실제 구현은 하나만 존재 : 타입 검사를 수행하고 어떤 함수를 사용할 지 결정
    def __mul__(self, other):
        if isinstance(other, Vector):
            return Vector(
                self.x * other.x,
                self.y * other.y
            )
        elif isinstance(other, (int, float)):
            return Vector(
                self.x * other,
                self.y * other
            )       

        return NotImplemented
 
    # --- 이 부분을 추가 작성 ---
    def __rmul__(self, other):
        # 숫자가 Vector를 곱하려 할 때, 결국 Vector가 곱셈의 주도권을 가져와서 
        # 위에서 정의한 __mul__을 다시 실행하게 합니다.
        return self.__mul__(other)


    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

v1 = Vector(1, 2)
v2 = Vector(3, 4)


print(v1*v2)
print(v1*3)


Vector(3, 8)
Vector(3, 6)


In [29]:
print(3*v1)

Vector(3, 6)
